# Choosing a model locally: cost, latency and limits with Ollama

By the end you can:

- run two local models on the same task and record what each one did;
- say what "cost" means when the model runs on your own laptop;
- explain the limits of local inference — memory, context, one model at a time;
- pick one model and defend the choice with numbers.

## Setup

**1. Install Ollama** from ollama.com — on Linux:

```bash
curl -fsSL https://ollama.com/install.sh | sh
ollama --version
```

**2. Pull both models** (about 10 GB together):

```bash
ollama pull qwen2.5:7b
ollama pull llama3.1:8b
ollama list
```

**3. One model in memory at a time.** Two 7B models do not both fit in 16 GB next to Docker and a browser:

```bash
export OLLAMA_MAX_LOADED_MODELS=1
export OLLAMA_CONTEXT_LENGTH=8192
export OLLAMA_KEEP_ALIVE=5m
```

On Windows and macOS these go in your user environment variables, then restart Ollama. On Linux, edit the systemd service and `sudo systemctl restart ollama`.

**4. Python side:**

```bash
uv pip install httpx
```

Ollama serves its API on `http://127.0.0.1:11434`. Nothing else needs installing — no key, no account, no card.

## The task, with the answers written first

Ten inbox emails, one question each: which queue does this belong to?

Write the expected answer **before** running anything. A label you invent after seeing the output is not a test.

Two of the ten are in Pidgin, on purpose. Local models are weaker on it, and you want that in your numbers, not hidden.

In [ ]:
EMAILS = [
    {"id": "E01", "expect": "billing",
     "subject": "Payment failed twice",
     "body": "My card was declined twice this morning. I need the invoice paid today."},
    {"id": "E02", "expect": "access",
     "subject": "Cannot log in after password reset",
     "body": "I reset my password and now the login page says my account is locked."},
    {"id": "E03", "expect": "general",
     "subject": "Question about the monthly plan",
     "body": "Does the basic plan include two users or only one?"},
    {"id": "E04", "expect": "urgent",
     "subject": "URGENT: production down",
     "body": "The API has returned 500 for forty minutes. Nobody can check out."},
    {"id": "E05", "expect": "billing",
     "subject": "Invoice 4402 is wrong",
     "body": "We were charged for eleven seats but only use seven."},
    {"id": "E06", "expect": "spam",
     "subject": "Congratulations you won",
     "body": "Click this link to claim your prize of five hundred thousand dollars."},
    {"id": "E07", "expect": "general",
     "subject": "Adding a teammate",
     "body": "How do I invite someone from my team to the workspace?"},
    {"id": "E08", "expect": "billing",
     "subject": "Dem don charge me two times",
     "body": "Abeg my card no dey work well, e don charge me two times this week."},
    {"id": "E09", "expect": "access",
     "subject": "Reporting tab no dey open",
     "body": "I be admin but the reports tab still dey tell me say I no get permission."},
    {"id": "E10", "expect": "general",
     "subject": "Meeting notes from Tuesday",
     "body": "Sharing the notes from our call in case anyone missed the decisions."},
]

print(len(EMAILS), "emails,", sum(1 for e in EMAILS if e["expect"] == "billing"), "of them billing")

## What makes a comparison fair

Four things must be identical across the two models, or the numbers mean nothing:

| Fixed | Why |
|---|---|
| the 10 emails | otherwise you compared prompts, not models |
| the prompt | one system prompt, one user message |
| `temperature: 0` | no randomness; the same input should give the same answer |
| `num_ctx: 8192` | context size changes both memory and speed |

And the expected answers were written before the first call. Change one of these and you have to run the comparison again.

## What "cost" means when nothing is billed

With Ollama there is no price per token. Your `ollama` bill is zero, forever.

That does not make cost disappear — it moves it:

| Where the cost went | How it shows up |
|---|---|
| electricity | a 7B model on CPU pulls the fan for a minute |
| your time | 10–30 minutes per comparison run |
| memory | 6 GB held while a model is loaded |
| opportunity cost | a weaker answer, if accuracy matters |

So this lesson keeps two cost numbers: **zero** (what you pay) and **what the same tokens would cost on a hosted API** (the number you quote when someone asks whether to move to the cloud). You need both to make the decision.

In [ ]:
MODELS = {
    "qwen2.5:7b": {"params": "7B", "download_gb": 4.7, "ram_gb": 5.7, "licence": "Apache 2.0"},
    "llama3.1:8b": {"params": "8B", "download_gb": 4.9, "ram_gb": 6.0,
                    "licence": "Meta Llama 3.1 Community"},
}

# what these exact tokens would cost on a hosted 8B-class API, USD per 1M tokens.
# EDIT and DATE this if you compare against a real provider.
HOSTED_REFERENCE = {"price_in": 0.05, "price_out": 0.08}

for name, info in MODELS.items():
    print(f"{name:<14} {info['params']:<4} {info['download_gb']} GB download, "
          f"{info['ram_gb']} GB in memory, {info['licence']}")

Read the memory column against your laptop, not against the other model.

`5.7 + 6.0 = 11.7 GB`. On a 16 GB machine that is fine for one model and not for two — which is exactly why `OLLAMA_MAX_LOADED_MODELS=1` is set. Without it, switching models makes Ollama try to hold both, and the machine swaps to disk or freezes.

The licence column matters too: Apache 2.0 and the Meta Community Licence are not the same rights. Check before reusing a model outside the course.

In [ ]:
import httpx

OLLAMA_TAGS = "http://127.0.0.1:11434/api/tags"
OLLAMA_CHAT = "http://127.0.0.1:11434/api/chat"

SYSTEM = ("You label inbox emails. Reply with exactly one word, one of: "
          "urgent, billing, access, general, spam.")


def build_prompt(email):
    return "Subject: " + email["subject"] + "\n\n" + email["body"]


def ask(model, prompt, fmt=None):
    payload = {
        "model": model,
        "messages": [{"role": "system", "content": SYSTEM},
                     {"role": "user", "content": prompt}],
        "stream": False,
        "options": {"temperature": 0, "num_ctx": 8192},
    }
    if fmt:
        payload["format"] = fmt

    # CPU inference is slow: allow two minutes, not two seconds
    r = httpx.post(OLLAMA_CHAT, json=payload, timeout=120)
    r.raise_for_status()
    d = r.json()

    return {
        "text": d["message"]["content"].strip(),
        "tokens_in": d.get("prompt_eval_count", 0),
        "tokens_out": d.get("eval_count", 0),
        "seconds": d.get("total_duration", 0) / 1e9,
    }

## The four numbers Ollama gives back

| Field | What it is | Unit |
|---|---|---|
| `prompt_eval_count` | your prompt + email, counted for you | tokens |
| `eval_count` | the answer | tokens |
| `total_duration` | whole call, model loading included | **nanoseconds** |
| `message.content` | the answer | text |

`total_duration` is in nanoseconds, which is why the wrapper divides by `1e9`. Skip that and every latency in your table is a billion times too large — a mistake that looks like a broken laptop.

You never count tokens by hand: Ollama reports both counters, so the numbers in your table are the model's own.

## One model at a time

Ollama keeps a model in memory for `OLLAMA_KEEP_ALIVE` (5 minutes by default) after its last call. Run two models in a row and, without a cap, it tries to hold both.

| Setting | Value | What it prevents |
|---|---|---|
| `OLLAMA_MAX_LOADED_MODELS=1` | one at a time | two 7B models in 16 GB |
| `OLLAMA_CONTEXT_LENGTH=8192` | modest context | memory climbing past your RAM |
| `OLLAMA_KEEP_ALIVE=5m` | unload after 5 min | a model held all afternoon |

The trap: a model that is **not** loaded yet pays a load time on its first call. That is why the next cell checks the server before measuring — and why the first call is thrown away.

In [ ]:
try:
    response = httpx.get(OLLAMA_TAGS, timeout=5)
    installed = [m["name"] for m in response.json()["models"]]
    print("server up --", len(installed), "models installed")
    for name in installed:
        print("  ", name)
    for name in MODELS:
        print(name, "ready" if name in installed else "MISSING -- run: ollama pull " + name)
except httpx.ConnectError:
    print("server not running -> open the Ollama app, or run: ollama serve")

## The limits of a local setup

Different limits from a hosted API, and they bite differently:

| Limit | Value here | What happens when you cross it |
|---|---|---|
| memory | ~6 GB per loaded model | swapping, then a frozen laptop |
| context | `num_ctx`, 8192 here | the answer ignores or drops the start of your prompt |
| concurrency | one request per model | a second request waits |
| throughput | roughly 3–15 tokens/sec on CPU | a long answer takes minutes |

There is no rate limit and no 429. Instead there is a queue, and the queue is your laptop. That is the trade: unlimited requests, limited speed.

## Latency: warm versus cold

Two different numbers hide in "how long did it take":

- **cold** — first call after a model was unloaded: load from disk into memory, then generate
- **warm** — model already loaded: generate only

On CPU the cold call can be several times slower. Report the warm number, because that is what a second email costs. Throw the first call away.

In [ ]:
import time


def timed_ask(model, prompt, runs=3):
    first = ask(model, prompt)            # warm-up: pays the model load, discarded

    times = []
    result = first
    for _ in range(runs):
        start = time.perf_counter()
        result = ask(model, prompt)
        times.append(time.perf_counter() - start)

    times.sort()
    result["warm_seconds"] = times[len(times) // 2]    # median
    result["slowest"] = times[-1]
    result["cold_seconds"] = first["seconds"]
    return result

In [ ]:
first = {}

for name in MODELS:
    email = EMAILS[0]
    cold = ask(name, build_prompt(email))        # model probably not loaded yet
    warm = ask(name, build_prompt(email))        # now it is
    first[name] = (cold["seconds"], warm["seconds"])
    print(f"{name:<14} cold {cold['seconds']:5.2f}s   warm {warm['seconds']:5.2f}s")

## What to record for every run

One row per call, six fields. Anything less and you cannot answer "why did you choose that model?" later.

| Field | From |
|---|---|
| the email id | your test set |
| the model's answer | `message.content` |
| matched the expected label? | your comparison |
| input tokens | `prompt_eval_count` |
| output tokens | `eval_count` |
| seconds | `total_duration / 1e9` |

The notebook keeps all six in `results`, which is why the table below can be rebuilt from it at any time.

## The comparison run

Ten emails, two models, three warm runs each. This is the slow cell — twenty minutes on a CPU-only laptop is normal. Plan it as an offline task, not something to start five minutes before a talk.

If you are testing on a weak machine, cut `runs` to 1 and say so in the note.

In [ ]:
results = {}

for name in MODELS:
    rows = []
    for email in EMAILS:
        r = timed_ask(name, build_prompt(email))
        r["id"] = email["id"]
        r["expect"] = email["expect"]
        r["match"] = r["text"].strip().lower().strip(".") == email["expect"]
        rows.append(r)
    results[name] = rows
    print(name, "-", len(rows), "runs,", sum(r["match"] for r in rows), "matched")

In [ ]:
def summarise(name):
    rows = results[name]
    warm = sorted(r["warm_seconds"] for r in rows)
    median = warm[len(warm) // 2]
    out_tokens = sum(r["tokens_out"] for r in rows) / len(rows)
    in_tokens = sum(r["tokens_in"] for r in rows) / len(rows)
    seconds_of_tokens = sum(r["seconds"] for r in rows)
    return {
        "median": median,
        "slowest": warm[-1],
        "in": in_tokens,
        "out": out_tokens,
        "tok_per_s": sum(r["tokens_out"] for r in rows) / seconds_of_tokens,
        "matches": sum(r["match"] for r in rows),
    }


summary = {name: summarise(name) for name in MODELS}

print(f"{'model':<14}{'median':>8}{'slowest':>9}{'tok/s':>7}{'in':>6}{'out':>6}{'matched':>9}")
for name, s in summary.items():
    print(f"{name:<14}{s['median']:>7.2f}s{s['slowest']:>8.2f}s{s['tok_per_s']:>7.1f}"
          f"{s['in']:>6.0f}{s['out']:>6.0f}{s['matches']:>6}/10")

## Reading the table

**Median** — what one more email costs once the model is warm.

**`tok/s`** — the honest local speed metric. It is roughly hardware, not model: the number moves with your CPU and RAM speed, and barely changes between two 7B models on the same laptop. Quote it so someone else can scale it to their machine.

**`matched`** — the column that decides. A model that is 30% faster and 20% wrong is not faster, it is a queue of tickets for a human.

Speed and cost decide *whether* you can run it. Accuracy decides *whether you should*. Ten emails is enough to catch a model that is clearly worse, and not enough to measure a 5% difference.

In [ ]:
def hosted_cost(name):
    s = summary[name]
    one = s["in"] / 1e6 * HOSTED_REFERENCE["price_in"] + s["out"] / 1e6 * HOSTED_REFERENCE["price_out"]
    return one * 1000


print(f"{'model':<14}{'$ now':>9}{'$/1,000 emails hosted':>24}")
for name in MODELS:
    print(f"{name:<14}{'0.00':>9}{hosted_cost(name):>24.4f}")

## Zero now, but know the other number

The first column is what this costs you: **nothing**, in money.

The second is what the same tokens would cost on a hosted API. At this size it is under a cent per thousand emails — the honest conclusion is that **price is not the reason to run locally**. You run locally for privacy, for offline work, and for zero sign-up. You run hosted for speed and quality.

Keeping both numbers means you can say that in one sentence instead of arguing about it. And when the day comes that the team needs 50,000 emails, you already know the size of the bill.

## Ask for JSON, then validate it

Free-text answers are hard to use. Ollama can constrain a reply to valid JSON with `"format": "json"` — but valid JSON is not the same as *correct* JSON: the keys can be missing, or the category can be outside your list.

So: ask for JSON, parse it, and count the failures. Never trust the shape you asked for.

In [ ]:
import json

PROMPT_JSON = ("Return JSON only, in the shape "
               '{"category": "urgent|billing|access|general|spam"}. ')

parsed, broken = 0, 0

for name in MODELS:
    for email in EMAILS[:5]:
        r = ask(name, PROMPT_JSON + build_prompt(email), fmt="json")
        try:
            body = json.loads(r["text"])
            ok = body.get("category") in {"urgent", "billing", "access", "general", "spam"}
            parsed += 1
            if not ok:
                broken += 1
                print("  parsed but invalid:", name, email["id"], body)
        except json.JSONDecodeError:
            broken += 1
            print("  not JSON:", name, email["id"], r["text"][:60])

print("parsed:", parsed, "| unusable:", broken)

Two different failures, and both count:

- **not JSON** — the reply came back as prose. `format: "json"` makes this rare, not impossible.
- **JSON but wrong** — `{"cat": "urgent"}` when you asked for `category`. Parsing succeeds; your code breaks later.

That second one is next week's problem: validate with a schema, reject what does not fit, and retry. The habit to start now is counting failures instead of assuming there are none.

## The decision, and how to write it down

Two models, one laptop, ten emails. The note that closes this exercise is three lines:

```
Chose: qwen2.5:7b
Because: 9/10 matched vs 8/10, median 6.2s vs 7.4s, both loaded in 6 GB
Caveat: 10 emails, one CPU-only laptop, Pidgin cases 2 of 10
```

Something like that is a decision. "It felt better" is not, and neither is a screenshot of a benchmark table from the internet.

The caveat line is not modesty — it tells the next person exactly when to re-run this. For this team, that is Week 14.

## If your laptop is the weak one

`qwen2.5:7b` needs about 5.7 GB at an 8K context, and Docker plus VS Code plus a browser take their share. On 8 GB, use a 4B model for labs and say so in your note — a documented 4B result is worth more than a comparison you could not finish.

The alternative is sharing one machine:

- on the laptop with the models, set `OLLAMA_HOST=0.0.0.0:11434` and restart Ollama
- everyone else calls `http://<that-ip>:11434/api/chat`

**Ollama has no login.** Anyone who can reach port 11434 can use it and read your prompts. Private network only, and set `OLLAMA_HOST` back to `127.0.0.1:11434` when the session ends.

## When it goes wrong

| Symptom | Cause | Fix |
|---|---|---|
| `connection refused` on 11434 | server not running | open the Ollama app, or `ollama serve` |
| `address already in use` | it is already running | quit the app; don't also run `ollama serve` |
| `model not found` | not pulled, or tag misspelled | `ollama pull qwen2.5:7b`, then `ollama list` |
| replies cut off mid-thought | `num_ctx` too small | raise it, or shorten the prompt |
| laptop freezes | two models loaded | `OLLAMA_MAX_LOADED_MODELS=1`, close Docker |
| very slow | CPU-only, entirely normal | `ollama ps` showing `100% CPU` is expected |

The last row is the one people misread: slow is the default on a laptop without a GPU, not a bug to fix.

## Summary

| Idea | Rule |
|---|---|
| Fair comparison | same emails, same prompt, `temperature: 0`, same `num_ctx` |
| Expected answers | written before the run, never after |
| Tokens | `prompt_eval_count` and `eval_count` from the response |
| Latency | warm median of 3; the cold call pays the model load |
| Speed metric | tokens per second, because it travels between machines |
| Cost | zero to you; quote the hosted equivalent |
| Limits | memory, context, one model at a time — no 429, just a queue |
| Accuracy | 10 emails finds the clearly worse model, not a 5% gap |
| Decision | model, reason, caveat — three lines, re-run at Week 14 |

One line: **a local model costs no money and all of your patience — measure both, then choose.**